# 01 Probe set
Run 10 probe tasks 5 times on two models and compare.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: pick two models

In [2]:
from kit.llm import call_llm, offline
from kit.prices import cost
import json, re
import pandas as pd

MODEL_A = os.environ.get("PROBE_MODEL_A", "gemini-3.5-flash-lite")
MODEL_B = os.environ.get("PROBE_MODEL_B", "gemini-3.6-flash")
if offline():
    MODEL_A, MODEL_B = "mock-model-a", "mock-model-b"
RUNS = 5
print(MODEL_A, "vs", MODEL_B)

mock-model-a vs mock-model-b


Step 3: load the probes and their pass criteria

The kit's ten probes are examples. Rewrite them in `data/probes.csv` (or in this cell) as ten tasks for your own product's help job. Each row's `check_type` and `check_value` is its pass criterion: write them all before you run anything below.

In [3]:
probes = pd.read_csv(DATA / "probes.csv").fillna("")
probes

,probe_id,task,check_type,check_value
0,p01,A user asks: How do I export a note as PDF? An...,must_contain,Settings
1,p02,"Reply with JSON only, no other text: an object...",valid_json,
2,p03,A user asks: Where is dark mode? Answer in one...,must_contain,steps
3,p04,"Classify this message as billing, bug or howto...",valid_json,
4,p05,Write a one line answer to: Can I add a tag? I...,regex,^(Mock|Yes)
5,p06,Summarise in under 30 words: My notes stopped ...,regex,"^.{1,240}$"
6,p07,A user asks: Can I share a notebook with my te...,must_contain,open
7,p08,Return JSON only with a key answer holding a s...,valid_json,
8,p09,A user writes: Why was I charged twice? Reply ...,must_contain,save
9,p10,Give a short answer to: Can I search inside im...,regex,^(?![\s\S]*[Ss]orry)[\s\S]+$


Step 4: define the pass checks

In [4]:
def passes(text, check_type, check_value):
    if check_type == "must_contain":
        return check_value in text
    if check_type == "regex":
        return re.search(check_value, text.strip(), re.S) is not None
    if check_type == "valid_json":
        try:
            json.loads(text.strip())
            return True
        except ValueError:
            return False
    raise ValueError(f"Unknown check type {check_type}")

print(passes('{"ok": true}', "valid_json", ""), passes("hi", "must_contain", "bye"))

True False


Step 5: run every probe 5 times on each model

In [5]:
rows = []
for model in [MODEL_A, MODEL_B]:
    for p in probes.itertuples():
        for run in range(1, RUNS + 1):
            r = call_llm(p.task, model=model, temperature=0.7, max_output_tokens=300)
            rows.append({"model": model, "probe_id": p.probe_id, "run": run,
                         "check_type": p.check_type, "check_value": p.check_value,
                         "passed": passes(r["text"], p.check_type, p.check_value),
                         "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
                         "stop_reason": r["stop_reason"],
                         "cost_usd": cost(r["input_tokens"], r["output_tokens"], model),
                         "latency_s": r["latency_s"], "text": r["text"][:300]})
runs = pd.DataFrame(rows)
runs.to_csv(OUTPUT / "probe_runs.csv", index=False)  # Stage 2 reads the failed runs from this file
print(len(runs), "calls, saved to", OUTPUT.name + "/probe_runs.csv")

100 calls, saved to stage-01/probe_runs.csv


Step 6: pass rate per probe

In [6]:
runs.pivot_table(index="probe_id", columns="model", values="passed", aggfunc="mean")

model,mock-model-a,mock-model-b
probe_id,,
p01,1.0,0.6
p02,1.0,0.6
p03,1.0,0.4
p04,1.0,0.6
p05,1.0,0.8
p06,1.0,1.0
p07,1.0,1.0
p08,1.0,0.6
p09,1.0,1.0


Step 7: compare the two models

In [7]:
summary = runs.groupby("model").agg(pass_rate=("passed", "mean"), cost_per_task=("cost_usd", "mean"),
                                    median_latency_s=("latency_s", "median"), calls=("passed", "size"))
summary["cost_per_task"] = summary["cost_per_task"].round(6)
summary.to_csv(OUTPUT / "probe_summary.csv")  # Stage 4 scores models with these numbers
summary

,pass_rate,cost_per_task,median_latency_s,calls
model,,,,
mock-model-a,0.90,0.000155,0.512,50
mock-model-b,0.68,0.000032,0.594,50


Step 8: save the files for later stages

`probe_runs.csv` (every run) and `probe_summary.csv` (the table above) are now in `stage-01/`. In Colab, download both from the Files panel: Stage 2 adds your failed probes to its backlog, and Stage 4 scores models with your numbers. Move them into `stage-01/` when you fork the kit in Chapter 3.

Step 9: five predicted weaknesses

Written before the runs, then checked against the tables above (example made with the offline mock model).

| # | Prediction, written before the runs | Layer | What the runs showed | Verdict |
| --- | --- | --- | --- | --- |
| 1 | The cheaper model breaks JSON output more often | model | p02, p04, p08: model B passed 3 of 5 each, model A 5 of 5 | holds |
| 2 | Both models ignore a "do not use the word X" rule | model | p10: model A 0 of 5, model B 1 of 5 | holds |
| 3 | Answers often hit the output cap and get cut off | harness | all 100 probe runs stopped with STOP at a 300 token cap; the longest used 46 tokens | false |
| 4 | The cheaper model sometimes gives a non answer | model | model B said "I am not sure." in 8 of 50 runs, model A in 0 | holds |
| 5 | With no help pages in the prompt, answers invent generic steps | context | every how-to answer gave the same "open Settings, pick the option, save" steps | holds, recheck in Stage 3 with help pages |

Disproved prediction, rewritten narrower: answers are cut off only when the output cap is far below the answer length, as in call 10 of `01_first_calls` (cap 10 tokens, stop reason MAX_TOKENS). At a 200 or 300 token cap no help answer was cut off.

Step 10: README

**Decisions:** Use model A for the help job for now: it passed 90 percent of probe runs against 68 percent for model B. Keep the output cap at 200 tokens. Do not use model B for JSON tasks until the harness checks and retries invalid JSON.

**What I measured:** 10 baseline calls (see `usage.csv`: about 29 input and 26 output tokens per call, $0.00134 in total, one MAX_TOKENS stop at a 10 token cap) and 100 probe runs, 10 tasks x 5 runs x 2 models (see `probe_summary.csv`): model A 90 percent pass, $0.000155 per task, 0.512 s median latency; model B 68 percent, $0.000032 per task, 0.594 s.

**What I would do next:** Rerun both notebooks with a real key, since these numbers come from the mock model. Add the help pages to the prompt and test prediction 5 again. Add a JSON check with one retry and rerun p02, p04 and p08 on model B.